In [ ]:
# Routed-GRL Full-Mamba HADA — BCI IV-2a LOSO (one cell). Enable GPU + Internet.
import codecs
import os
import re
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path

# ---------------- Experiment configuration ----------------
# STAGE 0: Full-Mamba baseline (rho = 1, no group path) -> must reproduce the old result
# STAGE 1: + (A) GRL routing, rho = clamp(backbone_gap / tau, 0, 1)
# STAGE 2: + (A) + (B) group-preserving residual path  (full method)
# STAGE 3: ablation, rho = 0 fixed (aligner never sees GRL; Lite-DA side bound)
STAGES = [2]
SEEDS = [0]
GRL_ROUTING_TAU = 1.0
TARGET_SUBJECTS = list(range(1, 10))
# One (stage, seed) run takes ~10 h for 9 targets: keep ONE stage and ONE seed
# per Kaggle session (12 h limit). Pick the stage that wins on Zhou2016.
DATASET = 'bcic2a'
BRANCH = 'feature/hada-full-mamba-routed-grl'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
REPO_PATH = Path(f"/kaggle/working/tcformer-routed-grl-{DATASET}")
MNE_DATA = Path("/kaggle/working/mne_data")
RUNS_DIR = Path(f"/kaggle/working/routed_grl_{DATASET}_runs")

STAGE_SETTINGS = {
    0: {'grl_routing': 'none', 'grl_aligner_rho': 1.0, 'group_aligner': False},
    1: {'grl_routing': 'gap', 'grl_aligner_rho': 1.0, 'group_aligner': False},
    2: {'grl_routing': 'gap', 'grl_aligner_rho': 1.0, 'group_aligner': True},
    3: {'grl_routing': 'fixed', 'grl_aligner_rho': 0.0, 'group_aligner': False},
}
assert all(stage in STAGE_SETTINGS for stage in STAGES), STAGES


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print("+", " ".join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    started_at = time.monotonic()
    with open(log_path, "w", encoding="utf-8") as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end="", flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f"[Kaggle heartbeat] running | elapsed={elapsed:.1f}m\n"
                print(heartbeat, end="", flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break

        remaining = decoder.decode(b"", final=True)
        if remaining:
            print(remaining, end="", flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Training exited with code {return_code}. Full log: {log_path}"
        )


# ---------------- Clone branch and verify the implementation ----------------
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
if (REPO_PATH / ".git").is_dir():
    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_PATH)
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_PATH)
    run(["git", "checkout", "--force", BRANCH], cwd=REPO_PATH)
    run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_PATH)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_PATH])
run(["git", "log", "-1", "--oneline"], cwd=REPO_PATH)

pipeline_text = (REPO_PATH / "train_pipeline.py").read_text(encoding="utf-8")
model_text = (REPO_PATH / "models/hada_tcformer.py").read_text(encoding="utf-8")
datamodule_text = (REPO_PATH / "utils/get_datamodule_cls.py").read_text(encoding="utf-8")
assert "ModelCheckpoint" in pipeline_text and 'monitor="val_acc"' in pipeline_text
assert "class GroupResidualAligner" in model_text and "_aligner_grl_share" in model_text
assert "zhou2016_loso" in datamodule_text
print("Verified: routed-GRL model, Zhou2016 support, best source-validation checkpoint.")

UV = shutil.which("uv") or "uv"
run([UV, "venv", "--clear", "--python", "3.10", ".venv"], cwd=REPO_PATH)
PYTHON = REPO_PATH / ".venv/bin/python"
run([UV, "pip", "install", "--python", PYTHON, "torch==2.7.1", "torchvision==0.22.1",
     "--index-url", "https://download.pytorch.org/whl/cu126"])
run([UV, "pip", "install", "--python", PYTHON, "-r", "requirements.txt"], cwd=REPO_PATH)
run(["nvidia-smi"])
run([PYTHON, "-c", "import torch; print('PyTorch:', torch.__version__); "
     "print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'; "
     "print('GPU:', torch.cuda.get_device_name(0))"])

MNE_DATA.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment.update({
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "MPLBACKEND": "Agg",
    "MNE_DATA": str(MNE_DATA),
    "MNE_DATASETS_BNCI_PATH": str(MNE_DATA),
    "MNE_DATASETS_ZHOU2016_PATH": str(MNE_DATA),
})

# ---------------- Train / test every (stage, seed) ----------------
summary_rows = []
for stage in STAGES:
    for seed in SEEDS:
        run_tag = f"{DATASET}_stage{stage}_seed{seed}"
        train_log = RUNS_DIR / f"{run_tag}.log"
        # Start every run from the committed config and a clean results folder.
        run(["git", "checkout", "--", "configs/hada_tcformer.yaml"], cwd=REPO_PATH)
        results_dir = REPO_PATH / "results"
        if results_dir.exists():
            shutil.rmtree(results_dir)
        settings = dict(STAGE_SETTINGS[stage], grl_routing_tau=GRL_ROUTING_TAU)
        override = (
            "import yaml; from pathlib import Path; "
            "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
            f"c['subject_ids']={TARGET_SUBJECTS!r}; "
            f"c['seed']={seed}; "
            "c['log_every_n_epochs']=10; "
            "c['enable_progress_bar']=False; "
            "c['model_kwargs']['log_every_n_batches']=0; "
            f"c['model_kwargs'].update({settings!r}); "
            "p.write_text(yaml.safe_dump(c, sort_keys=False))"
        )
        run([PYTHON, "-c", override], cwd=REPO_PATH)
        print(f"===== ROUTED-GRL | {DATASET} | STAGE={stage} {settings} | SEED={seed} "
              f"| TARGETS={TARGET_SUBJECTS} =====", flush=True)
        run([PYTHON, "-u", "train_pipeline.py", "--model", "hada_tcformer",
             "--dataset", DATASET, "--loso", "--gpu_id", "0"],
            cwd=REPO_PATH, env=environment, stream=True, log_path=train_log)
        shutil.move(str(results_dir), str(RUNS_DIR / run_tag))

        log_text = train_log.read_text(encoding="utf-8", errors="replace")
        accuracy = re.findall(r"Average Test Accuracy: ([0-9.]+ ± [0-9.]+)", log_text)
        summary_rows.append((stage, seed, accuracy[-1] if accuracy else "n/a"))
        # Last-epoch DA diagnostics per target subject (rho, gaps, D_acc, scales).
        da_lines = re.findall(r"\[DA\] epoch (\d+)/(\d+) \|(.*)", log_text)
        last_epoch_lines = [line for epoch, total, line in da_lines if epoch == total]
        for subject_id, line in zip(TARGET_SUBJECTS, last_epoch_lines):
            print(f"  S{subject_id:02d} final DA:{line}")

print("\n===== SUMMARY (primary test accuracy, mean ± std over subjects) =====")
for stage, seed, accuracy in summary_rows:
    print(f"STAGE {stage} | seed {seed} | {accuracy}")

archive = shutil.make_archive(str(RUNS_DIR), "zip", root_dir=RUNS_DIR.parent,
                              base_dir=RUNS_DIR.name)
print("Kaggle output archive:", archive)
